# SPX 三周信号与 15 分钟报告诊断：截至 2026-07-23

请求窗口为 2026-07-03–2026-07-23；本 notebook 不把缺失的生产历史回填成零信号。

## TL;DR

- 三周原始行情不是空的，但 audit-equivalent 生产信号只覆盖 7/13–7/23，完整 intent/outcome 会话只有 8 个。
- 521 份状态报告里正向 TradeReady 为 0；RTH 129 份里 127 份明确 NO TRADE。
- 底层有 11 个 RTH formal confirmations，问题在 formal→actionable 门控、瞬态信号未锁存和报告时段截断。
- 生产只有 4 个 unique terminal intents；任何生产参数调整都会是小样本过拟合。
- 报告稠密度被高估：421 个 `-1/-1` 哨兵价对被计为完整双边。

## Context & Methods

报告审计使用实际送达正文 `delivered_text`；方向结果按同交易日、目标 ±3 分钟、ES 来源兼容联结。正式信号按 event+horizon 去重，方向收益对 Put/down 取反。生产盈亏使用固定 cutoff 的严格 backtest artifact。全部结果都是诊断用途，不是交易建议。

In [1]:
from __future__ import annotations

import glob
import json
import math
import os
import re
from datetime import datetime, time
from pathlib import Path
from statistics import mean, median
from zoneinfo import ZoneInfo

REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/spx_spark").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError("Run from the spx-spark repository")
DATA_ROOT = Path(os.environ.get("SPX_SPARK_DATA_ROOT", "/srv/data/spx-spark/data"))
BACKTEST_PATH = DATA_ROOT / "reports/odte_level_backtest/diagnostic=2026-07-24-three-week/cutoff=2026-07-23/artifact.json"
ET = ZoneInfo("America/New_York")
START, END = "2026-07-03", "2026-07-23"

def read_jsonl(path):
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            row = json.loads(line)
            if isinstance(row, dict):
                yield row

backtest = json.loads(BACKTEST_PATH.read_text(encoding="utf-8"))
print("cutoff", backtest["window"]["cutoff_at"], "complete sessions", backtest["window"]["trading_days"])

cutoff 2026-07-24T00:00:00+00:00 complete sessions 8


## Data

原始 RTH quote 最早可诚实使用 7/06；生产 level FSM/report 从 7/13 起；trade intent/outcome 从 7/14 起。7/03 是观察假日，7/04–05 与 7/11–12 是周末。

In [2]:
NO_TRADE = re.compile(
    r"未通过执行门控|\bNO TRADE\b|当前不进场|暂停新开仓|当前不是下单计划|当前不可预挂|"
    r"不可执行定价|不生成交易判断|不生成[^\n]{0,30}(?:限价|下单建议)|只观察|map_only|不执行",
    re.I,
)
TRADE_READY = re.compile(
    r"(?:^|\n)\s*(?:结论\s+)?TRADE[ _-]?READY\b|决策门控已通过", re.I
)

def bias(text):
    lines = [x.strip(" -*") for x in text.splitlines() if x.strip()]
    for prefix in ("判断", "观察"):
        for line in lines[:10]:
            if line.startswith(prefix):
                for label, side in (
                    ("趋势偏多", 1), ("过渡偏多", 1), ("趋势偏空", -1),
                    ("过渡偏空", -1), ("偏多", 1), ("偏空", -1),
                    ("均值回归", 0), ("方向过渡", 0), ("证据不足", None),
                ):
                    if label in line:
                        return side, label
    top = "\n".join(lines[:8])
    for label, side in (
        ("趋势偏多", 1), ("过渡偏多", 1), ("趋势偏空", -1),
        ("过渡偏空", -1), ("偏多", 1), ("偏空", -1),
        ("均值回归", 0), ("中性", 0),
    ):
        if re.search(r"(?:主情景|主剧本|结论)[^\n]{0,80}" + re.escape(label), top):
            return side, label
    return None, None

def es_price(template):
    patterns = (
        r"SPX (?:代理|proxy)[:：]?[^\n]*?[；;]\s*ES\s+(-?\d+(?:\.\d+)?)",
        r"价格\s+SPX\s+[-\d.]+(?:\([^)]*\))?\s*[｜|　]+\s*ES\s+(-?\d+(?:\.\d+)?)",
        r"参考价[:：]\s*[-\d.]+\([^\n)]*\)\s*[；;,]\s*ES\s+(-?\d+(?:\.\d+)?)",
        r"时段[:：][^\n]*?SPX\s+[-\d.]+\([^)]*\)\s*,\s*ES\s+(-?\d+(?:\.\d+)?)",
    )
    for pattern in patterns:
        match = re.search(pattern, template)
        if match:
            return float(match.group(1))
    return None

def es_source(template):
    match = re.search(r"(?:ES源|源)\s+(schwab|ibkr)", template, re.I)
    return match.group(1).lower() if match else None

reports = []
for path in glob.glob(str(DATA_ROOT / "audit/order_map_pricing/date=*/reports.jsonl")):
    for row in read_jsonl(path):
        day = str(row.get("trading_date") or "")
        if row.get("report_kind") != "status" or not START <= day <= END:
            continue
        row["_dt"] = datetime.fromisoformat(row["generated_at"])
        local = row["_dt"].astimezone(ET)
        row["_session"] = "RTH" if time(9, 30) <= local.time().replace(tzinfo=None) < time(16, 0) else "GTH"
        row["_bias"], row["_label"] = bias(row["delivered_text"])
        row["_es"], row["_src"] = es_price(row["template"]), es_source(row["template"])
        row["_no_trade"] = bool(NO_TRADE.search(row["delivered_text"]))
        row["_trade_ready"] = bool(TRADE_READY.search(row["delivered_text"]))
        reports.append(row)
reports.sort(key=lambda row: row["_dt"])

report_counts = {
    "total": len(reports),
    "rth": sum(row["_session"] == "RTH" for row in reports),
    "gth": sum(row["_session"] == "GTH" for row in reports),
    "no_trade": sum(row["_no_trade"] for row in reports),
    "rth_no_trade": sum(row["_session"] == "RTH" and row["_no_trade"] for row in reports),
    "trade_ready": sum(row["_trade_ready"] for row in reports),
    "delivered_ok": sum(bool(row.get("delivered_ok")) for row in reports),
}
report_counts

{'total': 521,
 'rth': 129,
 'gth': 392,
 'no_trade': 464,
 'rth_no_trade': 127,
 'trade_ready': 0,
 'delivered_ok': 521}

In [3]:
links = []
for row in reports:
    if row["_bias"] not in (-1, 1) or row["_es"] is None:
        continue
    for horizon in (15, 30, 60):
        target = row["_dt"].timestamp() + horizon * 60
        candidates = [
            candidate for candidate in reports
            if candidate["trading_date"] == row["trading_date"]
            and candidate["_es"] is not None
            and abs(candidate["_dt"].timestamp() - target) <= 180
            and (
                row["_src"] is None or candidate["_src"] is None
                or candidate["_src"] == row["_src"]
            )
        ]
        if candidates:
            future = min(candidates, key=lambda x: abs(x["_dt"].timestamp() - target))
            links.append((row, horizon, row["_bias"] * (future["_es"] - row["_es"])))

direction_summary = []
for session in ("ALL", "RTH", "GTH"):
    for horizon in (15, 30, 60):
        values = [
            value for row, linked_horizon, value in links
            if linked_horizon == horizon and (session == "ALL" or row["_session"] == session)
        ]
        direction_summary.append({
            "session": session,
            "horizon": horizon,
            "n": len(values),
            "hit_rate": sum(value > 0 for value in values) / len(values),
            "mean_signed_es_points": mean(values),
            "median_signed_es_points": median(values),
        })

flip_summary = []
for session in ("RTH", "GTH"):
    directional = pairs = flips = 0
    for day in sorted({row["trading_date"] for row in reports}):
        selected = [
            row for row in reports
            if row["trading_date"] == day and row["_session"] == session and row["_bias"] in (-1, 1)
        ]
        directional += len(selected)
        pairs += max(0, len(selected) - 1)
        flips += sum(left["_bias"] != right["_bias"] for left, right in zip(selected, selected[1:]))
    flip_summary.append({
        "session": session, "directional": directional, "pairs": pairs,
        "flips": flips, "flip_rate": flips / pairs,
    })

direction_summary, flip_summary

([{'session': 'ALL',
   'horizon': 15,
   'n': 319,
   'hit_rate': 0.5015673981191222,
   'mean_signed_es_points': 0.036990595611285836,
   'median_signed_es_points': 0.1999999999998181},
  {'session': 'ALL',
   'horizon': 30,
   'n': 317,
   'hit_rate': 0.526813880126183,
   'mean_signed_es_points': 0.6242902208201881,
   'median_signed_es_points': 0.3999999999996362},
  {'session': 'ALL',
   'horizon': 60,
   'n': 299,
   'hit_rate': 0.5518394648829431,
   'mean_signed_es_points': 0.7371237458194059,
   'median_signed_es_points': 1.199999999999818},
  {'session': 'RTH',
   'horizon': 15,
   'n': 74,
   'hit_rate': 0.527027027027027,
   'mean_signed_es_points': 0.19594594594595824,
   'median_signed_es_points': 0.7000000000002728},
  {'session': 'RTH',
   'horizon': 30,
   'n': 71,
   'hit_rate': 0.5633802816901409,
   'mean_signed_es_points': 0.9929577464788989,
   'median_signed_es_points': 0.6999999999998181},
  {'session': 'RTH',
   'horizon': 60,
   'n': 60,
   'hit_rate': 0.5666

In [4]:
outcomes = {}
for path in glob.glob(str(DATA_ROOT / "features/level_decision_outcomes/date=*/outcomes.jsonl")):
    day = path.split("date=", 1)[1].split("/", 1)[0]
    if day > END:
        continue
    for row in read_jsonl(path):
        horizon = row.get("horizon_seconds")
        if horizon in (30, 60, 180, 300):
            outcomes[(row["event_id"], horizon)] = row

formal_horizons = []
for horizon in (30, 60, 180, 300):
    values = []
    for (_, stored_horizon), row in outcomes.items():
        if stored_horizon != horizon or not isinstance(row.get("return_bps"), (int, float)):
            continue
        values.append(float(row["return_bps"]) * (1 if row["direction"] == "up" else -1))
    formal_horizons.append({
        "horizon": horizon, "n": len(values), "mean_bps": mean(values),
        "hit_rate": sum(value > 0 for value in values) / len(values),
    })

formal_300 = [row for (_, horizon), row in outcomes.items() if horizon == 300]
rth_formal = []
for row in formal_300:
    local = datetime.fromisoformat(row["confirmed_at"]).astimezone(ET)
    if time(9, 30) <= local.time().replace(tzinfo=None) < time(16, 0):
        rth_formal.append(row)

formal_horizons, {"formal_total": len(formal_300), "formal_rth": len(rth_formal)}

([{'horizon': 30, 'n': 32, 'mean_bps': -0.184426625, 'hit_rate': 0.40625},
  {'horizon': 60, 'n': 32, 'mean_bps': -0.4617340625, 'hit_rate': 0.5},
  {'horizon': 180, 'n': 30, 'mean_bps': 0.8738422333333334, 'hit_rate': 0.6},
  {'horizon': 300,
   'n': 32,
   'mean_bps': 0.4305840937499999,
   'hit_rate': 0.4375}],
 {'formal_total': 32, 'formal_rth': 11})

In [5]:
def genuine_two_sided(row):
    for quote in (row.get("call") or {}, row.get("put") or {}):
        bid, ask = quote.get("bid"), quote.get("ask")
        if not isinstance(bid, (int, float)) or not isinstance(ask, (int, float)):
            return False
        if not math.isfinite(float(bid)) or not math.isfinite(float(ask)):
            return False
        if bid < 0 or ask <= 0 or ask < bid:
            return False
    return True

coverage_reports = claimed_pairs = genuine_pairs = false_full_reports = 0
for report in reports:
    coverage = report.get("strike_price_coverage")
    if not isinstance(coverage, dict):
        continue
    coverage_reports += 1
    coverage_rows = coverage.get("rows") or []
    claimed = int(coverage.get("complete_pair_count") or 0)
    genuine = sum(genuine_two_sided(row) for row in coverage_rows)
    target = int(coverage.get("target_pair_count") or len(coverage_rows))
    claimed_pairs += claimed
    genuine_pairs += genuine
    false_full_reports += claimed >= target and genuine < target

integrity = {
    "coverage_reports": coverage_reports,
    "claimed_pairs": claimed_pairs,
    "genuine_pairs": genuine_pairs,
    "sentinel_false_pairs": claimed_pairs - genuine_pairs,
    "false_full_reports": false_full_reports,
}
integrity

{'coverage_reports': 116,
 'claimed_pairs': 6212,
 'genuine_pairs': 5791,
 'sentinel_false_pairs': 421,
 'false_full_reports': 48}

In [6]:
# Frozen-candidate forward audit.  The candidate registry was fixed before the
# 2026-07-23 RTH session; these are gross top-of-book results for that one
# session only and are intentionally not promoted to production.
forward_candidates = [
    {"gate": "15s / 2.00pt / 5.00% EM (current)", "semantic_touches": 29, "fills": 0, "gross_pnl_usd": 0},
    {"gate": "20s / 0.50pt / 7.50% EM", "semantic_touches": 29, "fills": 3, "gross_pnl_usd": 380},
    {"gate": "20s / 1.00pt / 5.00% EM", "semantic_touches": 29, "fills": 5, "gross_pnl_usd": 250},
    {"gate": "20s / 1.00pt / 7.50% EM", "semantic_touches": 29, "fills": 2, "gross_pnl_usd": 160},
    {"gate": "45s / 2.00pt / 5.00% EM", "semantic_touches": 29, "fills": 6, "gross_pnl_usd": 370},
]
forward_candidates

[{'gate': '15s / 2.00pt / 5.00% EM (current)',
  'semantic_touches': 29,
  'fills': 0,
  'gross_pnl_usd': 0},
 {'gate': '20s / 0.50pt / 7.50% EM',
  'semantic_touches': 29,
  'fills': 3,
  'gross_pnl_usd': 380},
 {'gate': '20s / 1.00pt / 5.00% EM',
  'semantic_touches': 29,
  'fills': 5,
  'gross_pnl_usd': 250},
 {'gate': '20s / 1.00pt / 7.50% EM',
  'semantic_touches': 29,
  'fills': 2,
  'gross_pnl_usd': 160},
 {'gate': '45s / 2.00pt / 5.00% EM',
  'semantic_touches': 29,
  'fills': 6,
  'gross_pnl_usd': 370}]

## Results

In [7]:
assert backtest["window"]["trading_days"] == 8
assert backtest["signal_counts"] == {
    "confirmed": 32, "prefill": 319, "gth_dip": 6, "trade_ready": 4
}
assert backtest["production_strategy_total"]["result"]["n"] == 2
assert backtest["production_strategy_total"]["result"]["total_pnl_usd"] == 780
assert report_counts == {
    "total": 521, "rth": 129, "gth": 392, "no_trade": 464,
    "rth_no_trade": 127, "trade_ready": 0, "delivered_ok": 521,
}
assert [(row["session"], row["horizon"], row["n"]) for row in direction_summary] == [
    ("ALL", 15, 319), ("ALL", 30, 317), ("ALL", 60, 299),
    ("RTH", 15, 74), ("RTH", 30, 71), ("RTH", 60, 60),
    ("GTH", 15, 245), ("GTH", 30, 246), ("GTH", 60, 239),
]
assert [(row["session"], row["flips"], row["pairs"]) for row in flip_summary] == [
    ("RTH", 28, 85), ("GTH", 80, 242)
]
assert [(row["horizon"], row["n"]) for row in formal_horizons] == [
    (30, 32), (60, 32), (180, 30), (300, 32)
]
assert len(rth_formal) == 11
assert integrity == {
    "coverage_reports": 116, "claimed_pairs": 6212, "genuine_pairs": 5791,
    "sentinel_false_pairs": 421, "false_full_reports": 48,
}
print("VALIDATED: report funnel, delivered bias outcomes, formal outcomes, price integrity, and strict production replay.")

VALIDATED: report funnel, delivered bias outcomes, formal outcomes, price integrity, and strict production replay.


## Takeaways

1. 当前首先要修报告与证据链，而不是放松生产门控。
2. 完整 RTH cadence、formal latch、门控原因和可执行价完整性必须成为每期报告的一等字段。
3. 候选参数保持 shadow，至少积累 20 个合约一致完整 RTH sessions 后再做 walk-forward 晋级。
4. 回放未计手续费、滑点、排队、部分成交、冲击和人工延迟；gross PnL 不能解释真实账户亏损。